In [4]:

from pathlib import Path

DRIVE = Path("/content/gdrive/MyDrive")
target = "IITD_preprocessed_metadata.csv"

print("Searching Google Drive for:", target)
found = []

for p in DRIVE.rglob(target):
    if p.is_file():
        found.append(p)
        print("\nFOUND:", p)
        print("Size: {:.2f} KB".format(p.stat().st_size / 1024))

if not found:
    print("\nCSV not found anywhere in the currently mounted My Drive.")

print("\nSearching for the Notebook 02 output folder:")
for p in DRIVE.rglob("notebook02_preprocessing"):
    if p.is_dir():
        print("FOUND FOLDER:", p)
        for f in p.iterdir():
            print(" -", f.name)


Searching Google Drive for: IITD_preprocessed_metadata.csv

CSV not found anywhere in the currently mounted My Drive.

Searching for the Notebook 02 output folder:


In [5]:

from pathlib import Path

folder = Path(
    "/content/gdrive/MyDrive/Ocular_Project/results/"
    "IITD/notebook02_preprocessing"
)

print("Folder exists:", folder.exists())
print("Is directory:", folder.is_dir())

if folder.is_dir():
    print("\nContents:")
    items = list(folder.iterdir())

    if items:
        for p in items:
            print(
                "DIR " if p.is_dir() else "FILE",
                p.name,
                f"({p.stat().st_size:,} bytes)" if p.is_file() else ""
            )
    else:
        print("Folder is empty.")
else:
    print("\nParent folder contents:")
    parent = folder.parent
    if parent.exists():
        for p in parent.iterdir():
            print(p.name)


Folder exists: False
Is directory: False

Parent folder contents:
notebook03_quality_assessment


In [8]:

from google.colab import drive
from pathlib import Path

# Use a fresh mount point; don't delete or overwrite existing files.
MOUNT = "/content/drive2"
drive.mount(MOUNT)

PROJECT = Path(MOUNT) / "MyDrive/Ocular_Project"

print("\nProject exists:", PROJECT.exists())

if PROJECT.exists():
    print("Project contents:")
    for p in PROJECT.iterdir():
        print(("DIR " if p.is_dir() else "FILE "), p.name)

    folder = PROJECT / "results/IITD/notebook02_preprocessing"
    print("\nNotebook 02 output folder exists:", folder.exists())

    if folder.is_dir():
        for p in folder.iterdir():
            print(" -", p.name)


Mounted at /content/drive2

Project exists: True
Project contents:
DIR  datasets
DIR  models
DIR  generated_images
DIR  results

Notebook 02 output folder exists: True
 - images_224
 - IITD_preprocessed_metadata.csv
 - IITD_preprocessing_errors.csv


In [10]:

print("quality_df shape:", quality_df.shape)
print("quality_df columns:", quality_df.columns.tolist())
print("results count:", len(results))
print("errors count:", len(errors))

if errors:
    print("\nFirst 5 errors:")
    for e in errors[:5]:
        print(e)

if results:
    print("\nFirst result:")
    print(results[0])


quality_df shape: (0, 0)
quality_df columns: []
results count: 0
errors count: 3270

First 5 errors:
{'original_relative_path': '001/01_L.bmp', 'error': "[Errno 2] No such file or directory: '/content/gdrive/MyDrive/Ocular_Project/results/IITD/notebook02_preprocessing/images_224/001/01_L.png'"}
{'original_relative_path': '001/02_L.bmp', 'error': "[Errno 2] No such file or directory: '/content/gdrive/MyDrive/Ocular_Project/results/IITD/notebook02_preprocessing/images_224/001/02_L.png'"}
{'original_relative_path': '001/03_L.bmp', 'error': "[Errno 2] No such file or directory: '/content/gdrive/MyDrive/Ocular_Project/results/IITD/notebook02_preprocessing/images_224/001/03_L.png'"}
{'original_relative_path': '001/04_L.bmp', 'error': "[Errno 2] No such file or directory: '/content/gdrive/MyDrive/Ocular_Project/results/IITD/notebook02_preprocessing/images_224/001/04_L.png'"}
{'original_relative_path': '001/05_L.bmp', 'error': "[Errno 2] No such file or directory: '/content/gdrive/MyDrive/Ocul

In [11]:

from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import json

# Correct, verified Drive mount
PROJECT = Path("/content/drive2/MyDrive/Ocular_Project")
IN_CSV = PROJECT / "results/IITD/notebook02_preprocessing/IITD_preprocessed_metadata.csv"
OUT_DIR = PROJECT / "results/IITD/notebook03_quality_assessment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(IN_CSV)

# Correct stale paths from the previous Drive mount
df["processed_path"] = df["processed_path"].str.replace(
    "/content/gdrive/MyDrive/",
    "/content/drive2/MyDrive/",
    regex=False
)

# Verify the paths before processing
missing = df.loc[
    ~df["processed_path"].map(lambda p: Path(p).is_file()),
    "processed_path"
]
if not missing.empty:
    print("Missing image paths:", len(missing))
    print(missing.head().to_list())
    raise FileNotFoundError("Fix missing image paths before assessing quality.")

print("Input images:", len(df))
print("All processed image paths verified.")

def assess_quality(path):
    with Image.open(path) as im:
        arr = np.asarray(im.convert("L"), dtype=np.float32)

    brightness = float(arr.mean())
    contrast = float(arr.std())

    lap = (
        -4 * arr
        + np.roll(arr, 1, axis=0)
        + np.roll(arr, -1, axis=0)
        + np.roll(arr, 1, axis=1)
        + np.roll(arr, -1, axis=1)
    )

    sharpness = float(lap.var())
    noise_proxy = float(
        np.median(np.abs(lap - np.median(lap))) / 0.6745
    )

    return brightness, contrast, sharpness, noise_proxy

results, errors = [], []

for row in tqdm(df.itertuples(index=False), total=len(df),
                desc="Assessing image quality"):
    try:
        b, c, s, n = assess_quality(row.processed_path)
        results.append({
            "original_relative_path": row.original_relative_path,
            "processed_path": row.processed_path,
            "subject": row.subject,
            "eye": row.eye,
            "brightness_mean": b,
            "contrast_std": c,
            "sharpness_laplacian_variance": s,
            "noise_proxy": n
        })
    except Exception as e:
        errors.append({
            "original_relative_path": row.original_relative_path,
            "error": str(e)
        })

quality_df = pd.DataFrame(results)
metric_cols = [
    "brightness_mean", "contrast_std",
    "sharpness_laplacian_variance", "noise_proxy"
]

# Save metrics and errors
quality_df.to_csv(OUT_DIR / "IITD_image_quality_metrics.csv", index=False)
pd.DataFrame(errors).to_csv(OUT_DIR / "IITD_quality_errors.csv", index=False)

if not quality_df.empty:
    summary = quality_df[metric_cols].describe().round(4)
    summary.to_csv(OUT_DIR / "IITD_quality_summary.csv")

    flags = pd.DataFrame({
        "original_relative_path": quality_df["original_relative_path"]
    })
    for col in metric_cols:
        q1, q3 = quality_df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        flags[col + "_outlier"] = (
            (quality_df[col] < q1 - 1.5 * iqr) |
            (quality_df[col] > q3 + 1.5 * iqr)
        )
    flags.to_csv(OUT_DIR / "IITD_quality_outlier_flags.csv", index=False)
else:
    raise RuntimeError(
        "No quality metrics were produced. Inspect IITD_quality_errors.csv."
    )

with open(OUT_DIR / "IITD_quality_assessment_summary.json", "w") as f:
    json.dump({
        "input_images": len(df),
        "images_assessed": len(quality_df),
        "errors": len(errors),
        "image_dimensions": "224x224",
        "image_mode": "grayscale",
        "metrics": metric_cols,
        "note": "Noise is an exploratory proxy; outliers are dataset-relative."
    }, f, indent=2)

print("\nNOTEBOOK 03 RESULTS")
print("Images assessed:", len(quality_df))
print("Errors:", len(errors))
print("Output folder:", OUT_DIR)
display(summary)

print("\nGenerated files:")
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print(p.name)


Input images: 3270
All processed image paths verified.


Assessing image quality:   0%|          | 0/3270 [00:00<?, ?it/s]


NOTEBOOK 03 RESULTS
Images assessed: 3270
Errors: 0
Output folder: /content/drive2/MyDrive/Ocular_Project/results/IITD/notebook03_quality_assessment


,brightness_mean,contrast_std,sharpness_laplacian_variance,noise_proxy
count,3270.0000,3270.0000,3270.0000,3270.0000
mean,114.6998,71.1029,6017.4978,31.2044
std,14.5404,12.4994,4335.9690,30.2056
min,72.6261,46.9159,298.5925,2.9652
25%,104.5819,61.1453,2671.6196,10.3781
50%,109.4276,66.8658,4091.5745,14.8258
75%,124.4731,85.9170,10016.5149,57.8206
max,164.2203,96.7088,19932.5996,131.9496



Generated files:
IITD_image_quality_metrics.csv
IITD_quality_assessment_summary.json
IITD_quality_errors.csv
IITD_quality_outlier_flags.csv
IITD_quality_summary.csv


## Conclusion

In this notebook, image quality assessment was performed on the preprocessed IIT Delhi Iris Database to characterize the visual properties of the images before ocular recognition experiments.

A total of **3,270 preprocessed images** were evaluated successfully, with **zero assessment errors**. Four metrics were calculated: mean brightness, intensity-based contrast, Laplacian variance as a sharpness indicator, and an exploratory noise proxy.

### Key Results

| Metric | Mean | Standard Deviation | Minimum | Maximum |
|---|---:|---:|---:|---:|
| Brightness | 114.6998 | 14.5404 | 72.6261 | 164.2203 |
| Contrast | 71.1029 | 12.4994 | 46.9159 | 96.7088 |
| Sharpness (Laplacian variance) | 6017.4978 | 4335.9690 | 298.5925 | 19932.5996 |
| Noise proxy | 31.2044 | 30.2056 | 2.9652 | 131.9496 |

### Generated Outputs

The assessment results were saved in `results/IITD/notebook03_quality_assessment/`:

- `IITD_image_quality_metrics.csv` — per-image quality measurements.
- `IITD_quality_summary.csv` — descriptive statistics for the quality metrics.
- `IITD_quality_outlier_flags.csv` — dataset-relative outlier flags for each metric.
- `IITD_quality_errors.csv` — records any failed assessments; no errors occurred.
- `IITD_quality_assessment_summary.json` — summary of the assessment run.

### Interpretation and Limitations

The results show variation in brightness, contrast, sharpness, and the noise proxy across the dataset. The relatively wide range of sharpness values indicates differences in the measured high-frequency image structure, although these differences do not automatically mean that an image is unsuitable for recognition.

The noise measure is an exploratory proxy rather than a calibrated estimate of sensor noise. Similarly, outlier flags identify statistically unusual values relative to this dataset; they do not independently establish that an image is defective.

All measurements were calculated on the **224 × 224 grayscale preprocessed images**, rather than the original images. Consequently, the results characterize the preprocessed dataset and may be affected by resizing and grayscale conversion.

### Next Steps

The quality measurements will support further dataset analysis and interpretation of the baseline ocular recognition results. In the next notebook, a baseline recognition model will be developed to establish a reference performance level before evaluating subsequent approaches.